In [ ]:
# FORWARD SELECTION (AIC/BIC) + AUTO VIF FILTER (low multicollinearity only)
# Works on pre-split, standardized data

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings, os

# ---------------------------
# 0) Config
# ---------------------------
PATH_X_TRAIN = "/content/DataByDayXTrain.csv"
PATH_X_TEST  = "/content/DataByDayXTest.csv"
PATH_Y_TRAIN = "/content/DataByDayYTrain.csv"
PATH_Y_TEST  = "/content/DataByDayYTest.csv"

TARGET_COL   = "avg_brs"
CRITERION    = "AIC"        # or "BIC"
VIF_THRESHOLD = 10.0        # remove features above this value

# ---------------------------
# 1) Load data (pre-split)
# ---------------------------
X_train = pd.read_csv(PATH_X_TRAIN)
X_test  = pd.read_csv(PATH_X_TEST)

for df_ in (X_train, X_test):
    for c in list(df_.columns):
        if c.lower().startswith("unnamed"):
            df_.drop(columns=[c], inplace=True)

y_train_raw = pd.read_csv(PATH_Y_TRAIN)
y_test_raw  = pd.read_csv(PATH_Y_TEST)

y_train = (y_train_raw[TARGET_COL] if TARGET_COL in y_train_raw.columns else y_train_raw.iloc[:, 0]).astype(float)
y_test  = (y_test_raw[TARGET_COL] if TARGET_COL in y_test_raw.columns else y_test_raw.iloc[:, 0]).astype(float)

missing_in_test  = set(X_train.columns) - set(X_test.columns)
missing_in_train = set(X_test.columns) - set(X_train.columns)
if missing_in_test or missing_in_train:
    raise ValueError(f"Feature mismatch.\nMissing in X_test: {missing_in_test}\nMissing in X_train: {missing_in_train}")
X_test = X_test[X_train.columns]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ---------------------------
# 2) Helpers
# ---------------------------
def _fit_ic(Xmat, yvec, criterion="AIC"):
    Xc = sm.add_constant(Xmat, has_constant="add")
    m = sm.OLS(yvec, Xc).fit()
    if criterion.upper() == "AIC":
        return m.aic, m
    else:
        return m.bic, m

def forward_selection(Xtrain, ytrain, criterion="AIC", max_features=None, verbose=True):
    remaining = list(Xtrain.columns)
    selected = []
    best_ic = np.inf
    best_model = None
    if max_features is None:
        max_features = len(remaining)

    for _ in range(min(max_features, len(remaining))):
        candidates = []
        for cand in remaining:
            try:
                cols = selected + [cand]
                ic, m = _fit_ic(Xtrain[cols], ytrain, criterion=criterion)
                candidates.append((ic, cand, m))
            except Exception:
                continue
        if not candidates:
            if verbose: print("No more candidates improve the criterion.")
            break

        candidates.sort(key=lambda t: t[0])
        best_new_ic, best_cand, best_new_model = candidates[0]

        if best_new_ic + 1e-8 < best_ic:
            selected.append(best_cand)
            remaining.remove(best_cand)
            best_ic = best_new_ic
            best_model = best_new_model
            if verbose: print(f"[Forward-{criterion}] + {best_cand} -> {criterion}={best_ic:.3f}")
        else:
            if verbose: print(f"[Forward-{criterion}] Stopped (no improvement).")
            break

    return selected, best_model

def compute_vif(Xdf):
    Xc = sm.add_constant(Xdf, has_constant="add")
    vif_vals = []
    for i, col in enumerate(Xc.columns):
        vif = variance_inflation_factor(Xc.values, i)
        vif_vals.append((col, vif))
    vif_df = pd.DataFrame(vif_vals, columns=["Feature", "VIF"])
    return vif_df[vif_df["Feature"] != "const"].sort_values("VIF", ascending=False).reset_index(drop=True)

# ---------------------------
# 3) Run forward selection
# ---------------------------
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sel, model = forward_selection(X_train, y_train, criterion=CRITERION, verbose=True)

print(f"\n[Forward-{CRITERION}] Selected features before VIF filtering: {sel}")

if not sel or model is None:
    raise RuntimeError("Forward selection did not select any features.")

# ---------------------------
# 4) VIF Filtering – Keep only low-VIF features
# ---------------------------
X_sel = X_train[sel].copy()
while True:
    vif_df = compute_vif(X_sel)
    high_vif = vif_df[vif_df["VIF"] > VIF_THRESHOLD]
    print("\n[VIF Check]")
    print(vif_df)
    if high_vif.empty:
        print("✅ All VIF values are below threshold.")
        break
    drop_feat = high_vif.iloc[0]["Feature"]
    print(f"⚠️  Dropping '{drop_feat}' due to high VIF ({high_vif.iloc[0]['VIF']:.2f})")
    sel.remove(drop_feat)
    X_sel = X_train[sel]
    if len(sel) == 0:
        raise RuntimeError("All features removed due to high VIF!")

print(f"\n✅ Final selected low-VIF features: {sel}")

# ---------------------------
# 5) Refit final model & evaluate
# ---------------------------
X_train_sel = X_train[sel]
X_test_sel  = X_test[sel]

final_model = sm.OLS(y_train, sm.add_constant(X_train_sel, has_constant="add")).fit()
print("\n[OLS Summary - Low-VIF Model]")
print(final_model.summary())

y_pred = final_model.predict(sm.add_constant(X_test_sel, has_constant="add"))

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n[Test metrics]")
print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

# ---------------------------
# 6) Save results
# ---------------------------
os.makedirs("/mnt/data", exist_ok=True)

coef = final_model.params.rename("coef").to_frame()
coef.index.name = "term"
coef.reset_index(inplace=True)
coef_path = f"/mnt/data/forward_{CRITERION}_lowVIF_coefficients.csv"
coef.to_csv(coef_path, index=False)

pred_path = f"/mnt/data/forward_{CRITERION}_lowVIF_predictions.csv"
pd.DataFrame({"y_test": y_test.values, "y_pred": y_pred}).to_csv(pred_path, index=False)

vif_path = f"/mnt/data/forward_{CRITERION}_finalVIF_table.csv"
compute_vif(X_train_sel).to_csv(vif_path, index=False)

sel_path = f"/mnt/data/forward_{CRITERION}_lowVIF_features.txt"
with open(sel_path, "w") as f:
    for s in sel: f.write(f"{s}\n")

print("\nSaved:")
print(f" - {coef_path}")
print(f" - {pred_path}")
print(f" - {vif_path}")
print(f" - {sel_path}")


Train: (90, 19), Test: (23, 19)
[Forward-AIC] + restless_periods -> AIC=58.861
[Forward-AIC] + rem_sleep_duration -> AIC=39.589
[Forward-AIC] + hrv -> AIC=23.361
[Forward-AIC] + Avg RT Correct Go (ms) -> AIC=5.165
[Forward-AIC] + Correct Go Trials -> AIC=-4.659
[Forward-AIC] + Avg RT Correct No-Go (ms) -> AIC=-7.086
[Forward-AIC] + high_activity_time -> AIC=-8.946
[Forward-AIC] + average_met_minutes -> AIC=-10.739
[Forward-AIC] + total_calories -> AIC=-17.236
[Forward-AIC] + equivalent_walking_distance -> AIC=-24.282
[Forward-AIC] + active_calories -> AIC=-27.911
[Forward-AIC] + high_activity_met_minutes -> AIC=-28.671
[Forward-AIC] + deep_sleep_duration -> AIC=-29.898
[Forward-AIC] Stopped (no improvement).

[Forward-AIC] Selected features before VIF filtering: ['restless_periods', 'rem_sleep_duration', 'hrv', 'Avg RT Correct Go (ms)', 'Correct Go Trials', 'Avg RT Correct No-Go (ms)', 'high_activity_time', 'average_met_minutes', 'total_calories', 'equivalent_walking_distance', 'active